# RQ1 — Predictive Accuracy of ML Models for Wildfire Risk
*Which model (Logistic Regression, Random Forest, XGBoost) gives the best accuracy–complexity trade-off for predicting wildfire occurrence?*

**Outputs:** `RQ1_model_comparison.csv`, `RQ1_roc_curves.pdf`

In [1]:

# ============================================================
# Shared data loading & preprocessing (Algerian Forest Fires)
# ============================================================
import os, glob, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300,
    "font.size": 11, "axes.titlesize": 13, "axes.labelsize": 11,
    "axes.grid": True, "grid.alpha": 0.3, "figure.autolayout": True,
})

OUTDIR = "/kaggle/working"          # on Kaggle this is the output folder
os.makedirs(OUTDIR, exist_ok=True)

def find_csv():
    """Find the Algerian Forest Fires csv anywhere under /kaggle/input."""
    cands = glob.glob("/kaggle/input/**/*.csv", recursive=True)
    if not cands:                   # local fallback
        cands = glob.glob("**/*.csv", recursive=True)
    # prefer a file whose name mentions 'algerian' or 'forest'
    for c in cands:
        n = os.path.basename(c).lower()
        if "algerian" in n or "forest" in n or "fire" in n:
            return c
    return cands[0]

def load_algerian():
    path = find_csv()
    print("Loading:", path)
    with open(path, "r", encoding="utf-8", errors="ignore") as fh:
        lines = fh.read().splitlines()
    # The raw UCI file mixes a title line, two region headers and a blank row,
    # so we parse it line by line rather than with a fixed-width csv reader.
    rows = [ln.split(",") for ln in lines]
    header = None
    region = "Bejaia"           # first block in the UCI file
    region_switched = False
    records, cols = [], None
    for r in rows:
        cells = [str(x).strip() for x in r]
        joined = " ".join(cells).lower()
        if "temperature" in joined and ("rh" in joined or "ws" in joined):
            cols = [c.strip() for c in cells if c.strip() != ""]
            header = cols
            if records and not region_switched:
                region = "Sidi-Bel Abbes"   # second header => second region
                region_switched = True
            continue
        if all(c == "" for c in cells):
            continue
        if "region" in joined or "dataset" in joined:   # title / region label line
            if "sidi" in joined:
                region = "Sidi-Bel Abbes"; region_switched = True
            continue
        if header is None:
            continue
        data_cells = [c for c in cells if c != ""]
        if len(data_cells) < len(cols):
            continue
        rec = dict(zip(cols, data_cells[:len(cols)]))
        rec["region"] = region
        records.append(rec)
    df = pd.DataFrame.from_records(records)
    return df

df = load_algerian()
df.columns = [c.strip().replace(" ", "_") for c in df.columns]

# Standardise the target column name
target_col = [c for c in df.columns if "class" in c.lower()]
target_col = target_col[0] if target_col else df.columns[-2]
df = df.rename(columns={target_col: "Classes"})

# Clean target -> binary (fire = 1, not fire = 0)
df["Classes"] = (df["Classes"].astype(str).str.strip().str.lower()
                 .str.replace(r"\s+", " ", regex=True))
df = df[df["Classes"].isin(["fire", "not fire"])].copy()
df["target"] = (df["Classes"] == "fire").astype(int)

# Numeric feature columns
FEATURES = ["Temperature", "RH", "Ws", "Rain",
            "FFMC", "DMC", "DC", "ISI", "BUI", "FWI"]
FEATURES = [f for f in FEATURES if f in df.columns]
for c in FEATURES:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.dropna(subset=FEATURES + ["target"]).reset_index(drop=True)

print("Shape:", df.shape, "| Fire:", int(df.target.sum()),
      "| Not fire:", int((1-df.target).sum()))
print("Regions:", df["region"].value_counts().to_dict())
X = df[FEATURES].copy()
y = df["target"].copy()


Loading: /kaggle/input/notebooks/sudhanshu432/eda-and-fe-algerian-forest-fires-dataset/Algerian_forest_fires_cleaned_dataset.csv
Shape: (243, 17) | Fire: 137 | Not fire: 106
Regions: {'Bejaia': 243}


In [2]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve)
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3,
                                          stratify=y, random_state=42)
scaler = StandardScaler().fit(X_tr)
X_tr_s, X_te_s = scaler.transform(X_tr), scaler.transform(X_te)

models = {
    "Logistic Regression": (LogisticRegression(max_iter=1000), True),
    "Random Forest": (RandomForestClassifier(n_estimators=300, random_state=42), False),
}
if HAS_XGB:
    models["XGBoost"] = (XGBClassifier(n_estimators=300, max_depth=4,
                         learning_rate=0.1, eval_metric="logloss",
                         use_label_encoder=False, random_state=42), False)

rows, roc_data = [], {}
for name, (clf, scale) in models.items():
    Xtr_, Xte_ = (X_tr_s, X_te_s) if scale else (X_tr.values, X_te.values)
    clf.fit(Xtr_, y_tr)
    prob = clf.predict_proba(Xte_)[:, 1]
    pred = (prob >= 0.5).astype(int)
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_te, pred),
        "Precision": precision_score(y_te, pred),
        "Recall": recall_score(y_te, pred),
        "F1": f1_score(y_te, pred),
        "ROC_AUC": roc_auc_score(y_te, prob),
    })
    fpr, tpr, _ = roc_curve(y_te, prob)
    roc_data[name] = (fpr, tpr, roc_auc_score(y_te, prob))

res = pd.DataFrame(rows).round(3)
res.to_csv(f"{OUTDIR}/RQ1_model_comparison.csv", index=False)
print(res.to_string(index=False))


              Model  Accuracy  Precision  Recall    F1  ROC_AUC
Logistic Regression     0.918      0.907   0.951 0.929    0.983
      Random Forest     0.986      0.976   1.000 0.988    1.000
            XGBoost     0.986      0.976   1.000 0.988    0.999


In [3]:

fig, ax = plt.subplots(figsize=(6, 5))
for name, (fpr, tpr, auc) in roc_data.items():
    ax.plot(fpr, tpr, lw=2, label=f"{name} (AUC = {auc:.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.6)
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("RQ1: ROC Curves — Wildfire Risk Classifiers")
ax.legend(loc="lower right", frameon=True)
fig.savefig(f"{OUTDIR}/RQ1_roc_curves.pdf", bbox_inches="tight")
print("Saved RQ1_roc_curves.pdf")


Saved RQ1_roc_curves.pdf
